In [4]:
%gui qt

In [5]:
import napari
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import napari
from sklearn.cluster import DBSCAN

In [6]:
me3_df = pd.read_csv("test_data/k27_k27_thaw009_me3.csv")
ac_df = pd.read_csv("test_data/k27_k27_thaw009_ac.csv")

In [7]:
print(me3_df.keys())
me3_df.head()

Index(['id', 'frame', 'x [nm]', 'y [nm]', 'sigmax [nm]', 'sigmay [nm]',
       'intensity [photon]', 'offset [photon]', 'uncertainty [nm]',
       'centroid [nm]', 'bgstd', 'z [nm]', 'z uncertainty', 'ydif',
       'centroid2', 'fwhm0', 'fwhm1'],
      dtype='object')


,id,frame,x [nm],y [nm],sigmax [nm],sigmay [nm],intensity [photon],offset [photon],uncertainty [nm],centroid [nm],bgstd,z [nm],z uncertainty,ydif,centroid2,fwhm0,fwhm1
0,1.0,1.0,13935.849519,6057.674068,149.42,149.42,2523.70,139.90,7.5735,673.60,0.0,714.0,67.916,0.20389,678.14,364.96,459.06
1,2.0,1.0,23512.849519,7335.074068,125.46,125.46,779.69,147.20,9.7751,668.54,0.0,321.0,88.617,-0.16929,676.19,453.47,299.69
2,3.0,1.0,18775.849519,13745.674068,142.93,142.93,2735.60,124.09,6.2818,668.29,0.0,640.0,56.257,1.41450,674.41,379.43,422.45
3,7.0,1.0,9219.849519,15277.674068,264.52,264.52,9646.30,314.73,13.9470,677.86,0.0,476.0,126.410,-1.69200,680.86,598.35,494.88
4,8.0,1.0,13670.849519,16081.674068,217.99,217.99,2811.90,364.75,18.7260,673.00,0.0,549.0,169.870,0.87030,677.45,670.82,632.57


In [8]:
me3_np = me3_df[["x [nm]", "y [nm]", "z [nm]"]].to_numpy()
me3_clust = DBSCAN(eps=50, min_samples=3, n_jobs=-1).fit_predict(me3_np)

me3_df["cluster_label"] = me3_clust

ac_np = ac_df[["x [nm]", "y [nm]", "z [nm]"]].to_numpy()
ac_clust = DBSCAN(eps=50, min_samples=3, n_jobs=-1).fit_predict(ac_np)

ac_df["cluster_label"] = ac_clust

In [10]:
rng = np.random.default_rng(0)  # reproducible colors
alpha = 0.5

me3_unique_clusters = np.unique(me3_clust[me3_clust != -1])
me3_cluster_colors = {c: np.append(rng.random(3), 1) for c in me3_unique_clusters}
me3_colors = []

for c in me3_clust:
    if c == -1:
        me3_colors.append((0.7, 0.7, 0.7, alpha))
    else:
        me3_colors.append(me3_cluster_colors[c])

ac_unique_clusters = np.unique(ac_clust[ac_clust != -1])
ac_cluster_colors = {c: np.append(rng.random(3), 1) for c in ac_unique_clusters}
ac_colors = []

for c in ac_clust:
    if c == -1:
        ac_colors.append((0.7, 0.7, 0.7, alpha))
    else:
        ac_colors.append(ac_cluster_colors[c])

me3_colors = np.array(me3_colors)
ac_colors = np.array(ac_colors)

In [ ]:
plt.figure(figsize=(12,12))

plt.scatter(me3_df["x [nm]"], me3_df["y [nm]"], c=me3_colors, s=1, linewidths=0)
plt.xlabel("x")
plt.ylabel("y")
plt.title("H3K27me3 Clusters")

plt.show()

In [ ]:
plt.figure(figsize=(12,12))

plt.scatter(ac_df["x [nm]"], ac_df["y [nm]"], c=ac_colors, s=1, linewidths=0)
plt.xlabel("x")
plt.ylabel("y")
plt.title("H3K27ac Clusters")

plt.show()

In [22]:
viewer = napari.Viewer()

me3_size = me3_df['sigmax [nm]'].to_numpy()/110*10
ac_size = ac_df['sigmax [nm]'].to_numpy()/110*10

me3_cluster_labels = me3_df['cluster_label']
ac_cluster_labels = ac_df['cluster_label']

me3_border = [0, 0.7, 0, 1]
ac_border = [0.7, 0, 0, 1]

viewer.add_points(ac_np[ac_cluster_labels == -1], face_color=ac_colors[ac_cluster_labels == -1], border_color=ac_border, border_width=0.01, size=ac_size[ac_cluster_labels == -1], name="H3K27ac Noise")                           
viewer.add_points(me3_np[me3_cluster_labels == -1], face_color=me3_colors[me3_cluster_labels == -1], border_color=me3_border, border_width=0.01, size=me3_size[me3_cluster_labels == -1], name="H3K27me3 Noise")
viewer.add_points(ac_np[ac_cluster_labels != -1], face_color=ac_colors[ac_cluster_labels != -1], border_color=ac_border, border_width=0.01, size=ac_size[ac_cluster_labels != -1], name="H3K27ac Nanodomains")
viewer.add_points(me3_np[me3_cluster_labels != -1], face_color=me3_colors[me3_cluster_labels != -1], border_color=me3_border, border_width=0.01, size=me3_size[me3_cluster_labels != -1], name="H3K27me3 Nanodomains")


<Points layer 'H3K27me3 Nanodomains' at 0x2051302e810>